In [78]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('ggplot')

In [79]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
from scipy.stats import skew
from sklearn.decomposition import PCA, KernelPCA
from xgboost import XGBRegressor

In [80]:
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.linear_model import ElasticNet, SGDRegressor, BayesianRidge
from sklearn.kernel_ridge import KernelRidge
from xgboost import XGBRegressor

In [81]:
pd.set_option('display.max_columns',500)
pd.set_option('display.max_rows',1000)

In [82]:
train=pd.read_csv('./X_train.csv')
test=pd.read_csv('./X_test.csv')
price_res=pd.read_csv('./y_train.csv')
train = pd.merge(train, price_res, on="Id") 

# data preprocessing

In [83]:
train .columns
for col in train.columns:
    print(col,train[col].dtype
           )

Id int64
鄉鎮市區 object
交易標的 object
路名 object
土地移轉總面積平方公尺 float64
都市土地使用分區 object
土地數 float64
建物數 float64
車位數 float64
移轉層次 int64
移轉層次項目 object
總樓層數 float64
建物型態 object
主要用途 object
主要建材 object
建築完成年月 object
建物移轉總面積平方公尺 float64
建物現況格局-房 int64
建物現況格局-廳 int64
建物現況格局-衛 int64
建物現況格局-隔間 object
有無管理組織 object
交易年 int64
交易日 int64
交易月 int64
地鐵站 float64
超商 float64
公園 float64
托兒所 float64
國小 float64
國中 float64
高中職 float64
大學 float64
金融機構 float64
醫院 float64
大賣場 float64
超市 float64
百貨公司 float64
警察局 float64
消防局 float64
縱坐標 float64
橫坐標 float64
單價元平方公尺 float64


In [117]:
full = train

### deal with years

In [118]:
# Convert '建築完成年月' to datetime format
full['建築完成年月'] = pd.to_datetime(full['建築完成年月'])

# Get the current date
today = datetime.today()

df = pd.DataFrame()


df['date_str'] = full['交易年'].astype(str) + '-' + full['交易月'].astype(str).str.zfill(2) + '-' + full['交易日'].astype(str).str.zfill(2)
df['交易日期'] = pd.to_datetime(df['date_str'])
#ˋ計算交易時屋齡
full['建築年紀']=df['交易日期'].dt.year- full['建築完成年月'].dt.year


print(full['建築年紀'])
#
full['交易距今時間'] = ((today.year - df['交易日期'].dt.year) * 12) + (today.month - df['交易日期'].dt.month)

print(full['交易距今時間'])


0        0
1       24
2       11
3       10
4       48
        ..
9455    46
9456    43
9457    19
9458     7
9459    12
Name: 建築年紀, Length: 9460, dtype: int32
0       63
1       46
2       55
3       65
4       43
        ..
9455    43
9456    56
9457    58
9458    60
9459    47
Name: 交易距今時間, Length: 9460, dtype: int32


#### drop original trade and build date

In [119]:
full = full.drop(['交易年', '交易日', '交易月','建築完成年月'], axis=1)

### encode

In [120]:
from sklearn.preprocessing import LabelEncoder

labelencoder = LabelEncoder()

for col in full.columns:
    if full[col].dtype=='object':
        full[col] = labelencoder.fit_transform(train[col])


# model & Evaluate

In [121]:
y_log= full['單價元平方公尺']

X_scaled=full

In [122]:
## drop id and price

full = full.drop(['單價元平方公尺', 'Id'], axis=1)

In [90]:
# define cross validation strategy
def rmse_cv(model,X,y):
    rmse = np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=5))
    return rmse

In [129]:
models = [RandomForestRegressor(500),
          ExtraTreesRegressor(),XGBRegressor()]

In [130]:
names = [ "RF","Extra","XGB"]
for name, model in zip(names, models):
    score = rmse_cv(model,full, train['單價元平方公尺'])
    print("{}: {:.6f}, {:.4f}".format(name,score.mean(),score.std()))

RF: 33296.125731, 651.2722
Extra: 33061.045449, 845.1796
XGB: 33395.612776, 809.3982


In [ ]:
class grid():
    def __init__(self,model):
        self.model = model
    
    def grid_get(self,X,y,param_grid):
        grid_search = GridSearchCV(self.model,param_grid,cv=5, scoring="neg_mean_squared_error")
        grid_search.fit(X,y)
        print(grid_search.best_params_, np.sqrt(-grid_search.best_score_))
        grid_search.cv_results_['mean_test_score'] = np.sqrt(-grid_search.cv_results_['mean_test_score'])
        print(pd.DataFrame(grid_search.cv_results_)[['params','mean_test_score','std_test_score']])

# ensemble

In [123]:
class AverageWeight(BaseEstimator, RegressorMixin):
    def __init__(self,mod,weight):
        self.mod = mod
        self.weight = weight
        
    def fit(self,X,y):
        self.models_ = [clone(x) for x in self.mod]
        for model in self.models_:
            model.fit(X,y)
        return self
    
    def predict(self,X):
        w = list()
        pred = np.array([model.predict(X) for model in self.models_])
        # for every data point, single model prediction times weight, then add them together
        for data in range(pred.shape[1]):
            single = [pred[model,data]*weight for model,weight in zip(range(pred.shape[0]),self.weight)]
            w.append(np.sum(single))
        return w

In [124]:
xgb=XGBRegressor()
rf=RandomForestRegressor()
extra=ExtraTreesRegressor()

In [125]:
# assign weights based on their gridsearch score
w1 = 0.4
w2 = 0.3
w3 = 0.3



In [ ]:
weight_avg = AverageWeight(mod = [rf,xgb,extra],weight=[w1,w2,w3])

In [132]:
score =rmse_cv(weight_avg,X_scaled,y_log),  rmse_cv(weight_avg,X_scaled,y_log).mean()

In [133]:
score

(array([376.45775618, 441.70004858, 413.95066436, 417.41205237,
        456.81958918]),
 np.float64(421.52426279606243))